In [36]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [37]:
# Hyperparameters
batch_size = 4     # num independent examples
block_size = 1024  # max sequence length
n_embd = 768       # total embedding dim, both in and out, divisible by n_head
n_head = 12        # number of heads
assert n_embd % n_head == 0
head_size = n_embd // n_head

# Init
torch.manual_seed(42)
c_attn_W = torch.randn(2304, 768) / 2304**0.5
c_attn_b = torch.randn(2304)
c_proj_W = torch.randn(768, 768) / 768**0.5
c_proj_b = torch.randn(768)
x = torch.randn(batch_size,block_size,n_embd)

In [38]:
# Reference Implementation

class CausalSelfAttentionMarcin(nn.Module):
    """Multiple self-attention heads"""
    def __init__(self, n_head, n_embd):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head

        self.c_attn = nn.Linear(n_embd, 3*n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1  # flag to scale proj into residual

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)  # B, T, nh*hs
        q = q.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        k = k.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        v = v.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        q = q.transpose(1, 2)  # B,nh,T,hs
        k = k.transpose(1, 2)  # B,nh,T,hs
        v = v.transpose(1, 2)  # B,nh,T,hs

        # W_affin = q @ k.mT / k.shape[-1]**0.5  # B,nh,T,hs @ B,nh,hs,T -> B,nh,T,T
        # W_affin = W_affin.masked_fill(self.bias[:,:,:T,:T]==0, float('-inf'))
        # W_affin = torch.softmax(W_affin, dim=-1)  # B,nh,T,T
        # y = W_affin @ v    # B,nh,T,T @ B,nh,T,hs -> B,nh,T,hs
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        y = y.transpose(1, 2)  # B,T,nh,hs
        y = y.contiguous()
        y = y.view(B,T,C)

        out = self.c_proj(y)
        return out

In [39]:
# Run reference implementation

csa_m = CausalSelfAttentionMarcin(n_head=n_head, n_embd=n_embd)
csa_m_state = csa_m.state_dict()
csa_m_state['c_attn.weight'] = c_attn_W.clone()
csa_m_state['c_attn.bias'] = c_attn_b.clone()
csa_m_state['c_proj.weight'] = c_proj_W.clone()
csa_m_state['c_proj.bias'] = c_proj_b.clone()
csa_m.load_state_dict(csa_m_state)

y_m2 = csa_m(x)
print(y_m2.shape)
print(y_m2.sum().item())

torch.Size([4, 1024, 768])
-18783.599609375


In [40]:
# RoPE Implementation (no class)

# Initialize linear projections
c_q = nn.Linear(n_embd, n_embd)
c_k = nn.Linear(n_embd, n_embd)
c_v = nn.Linear(n_embd, n_embd)
c_proj = nn.Linear(n_embd, n_embd)

with torch.no_grad():
    c_q.weight.copy_(c_attn_W[:n_embd])
    c_q.bias.copy_(c_attn_b[:n_embd])
    c_k.weight.copy_(c_attn_W[n_embd:2*n_embd])
    c_k.bias.copy_(c_attn_b[n_embd:2*n_embd])
    c_v.weight.copy_(c_attn_W[2*n_embd:])
    c_v.bias.copy_(c_attn_b[2*n_embd:])
    c_proj.weight.copy_(c_proj_W)
    c_proj.bias.copy_(c_proj_b)
    

In [41]:
# Calculate outputs w/o RoPE
with torch.no_grad():
    B, T, C = x.size()

    q = c_q(x)    # B, T, nh*hs
    k = c_k(x)    # B, T, nh*hs
    v = c_v(x)    # B, T, nh*hs
    q = q.view(B, T, n_head, C//n_head)  # B,T,nh,hs
    k = k.view(B, T, n_head, C//n_head)  # B,T,nh,hs
    v = v.view(B, T, n_head, C//n_head)  # B,T,nh,hs
    q = q.transpose(1, 2)  # B,nh,T,hs
    k = k.transpose(1, 2)  # B,nh,T,hs
    v = v.transpose(1, 2)  # B,nh,T,hs

    y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

    y = y.transpose(1, 2)  # B,T,nh,hs
    y = y.contiguous()
    y = y.view(B,T,C)

    out = c_proj(y)

    print(out.sum().item())

-18783.599609375


In [100]:
# d = head_size // 2
# print(d)
# i_range = torch.arange(1, d+1)  # +1 to make inclusive
# exp_range = (2*((i_range-1)/d))
# print(exp_range.shape)
# exp_range[:10]

In [ ]:
# Compute exponent for the RoPE frequencies
exp = torch.arange(0, head_size, step=2)
exp = exp/head_size    # no 2* because step=2 already
exp2 = 10_000**-exp
print(exp2.shape)
print(exp2[:6])
print(exp2[-4:])

torch.Size([32])
tensor([1.0000, 0.5623, 0.3162, 0.1778, 0.1000, 0.0562])
tensor([1.0000e-07, 5.6234e-08, 3.1623e-08, 1.7783e-08])


In [102]:
# Expand to pairwise
theta = exp2.repeat_interleave(2)
print(theta.shape)
print(theta[:6])
print(theta[-4:])

torch.Size([64])
tensor([1.0000, 1.0000, 0.5623, 0.5623, 0.3162, 0.3162])
tensor([3.1623e-08, 3.1623e-08, 1.7783e-08, 1.7783e-08])


In [103]:
# Positions
pos = torch.arange(0, 10*1024)  # overprovision to support longer sequences
pos.shape

torch.Size([10240])

In [104]:
# Multiply out
tmp = torch.outer(pos, theta)
print(tmp.shape)
tmp[:4, :8]

torch.Size([10240, 64])


tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 1.0000, 0.5623, 0.5623, 0.3162, 0.3162, 0.1778, 0.1778],
        [2.0000, 2.0000, 1.1247, 1.1247, 0.6325, 0.6325, 0.3557, 0.3557],
        [3.0000, 3.0000, 1.6870, 1.6870, 0.9487, 0.9487, 0.5335, 0.5335]])

In [105]:
# Compute sin and cos
sin = torch.sin(tmp)
cos = torch.cos(tmp)
print(sin.shape, cos.shape)
print(sin[:4, :8])
print(cos[:4, :8])

torch.Size([10240, 64]) torch.Size([10240, 64])
tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8415, 0.8415, 0.5332, 0.5332, 0.3110, 0.3110, 0.1769, 0.1769],
        [0.9093, 0.9093, 0.9021, 0.9021, 0.5911, 0.5911, 0.3482, 0.3482],
        [0.1411, 0.1411, 0.9933, 0.9933, 0.8126, 0.8126, 0.5085, 0.5085]])
tensor([[ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.5403,  0.8460,  0.8460,  0.9504,  0.9504,  0.9842,  0.9842],
        [-0.4161, -0.4161,  0.4315,  0.4315,  0.8066,  0.8066,  0.9374,  0.9374],
        [-0.9900, -0.9900, -0.1160, -0.1160,  0.5828,  0.5828,  0.8610,  0.8610]])


In [106]:
# import matplotlib.pyplot as plt
# plt.plot(sin[:100,0]);
# plt.plot(cos[:100,0]);
# plt.plot(sin[:100,100]);
# plt.plot(cos[:100,100]);

In [107]:
x[:2,:2,:4]

tensor([[[ 0.3200, -0.2651, -0.0264,  2.1537],
         [ 2.2608,  1.7447,  0.7011,  0.3407]],

        [[ 0.8515, -0.0897,  0.1958, -1.5051],
         [-1.2797, -1.5142, -0.0516,  0.5504]]])

In [108]:
B, T, C = x.size()

In [109]:
# Trim sin, cos to T and add batch dim
sin_trimmed = sin[:T, :].view(1, T, 1, head_size)
cos_trimmed = cos[:T, :].view(1, T, 1, head_size)
print(sin_trimmed.shape)
print(cos_trimmed.shape)

torch.Size([1, 1024, 1, 64])
torch.Size([1, 1024, 1, 64])


In [110]:
# Calculate outputs with RoPE
with torch.no_grad():
    q = c_q(x)    # B, T, nh*hs
    k = c_k(x)    # B, T, nh*hs
    v = c_v(x)    # B, T, nh*hs
    q = q.view(B, T, n_head, C//n_head)  # B,T,nh,hs
    k = k.view(B, T, n_head, C//n_head)  # B,T,nh,hs
    v = v.view(B, T, n_head, C//n_head)  # B,T,nh,hs



In [111]:
q.shape, k.shape, v.shape

(torch.Size([4, 1024, 12, 64]),
 torch.Size([4, 1024, 12, 64]),
 torch.Size([4, 1024, 12, 64]))

In [112]:
# Swap items at x_0, x_1 over last dim (pairwise swap)
q_even = q[:,:,:,0::2]
q_odd = q[:,:,:,1::2]
q_swapped = torch.stack((q_odd, q_even), dim=-1).reshape_as(q)
print(q_swapped.shape)
q_swapped[:2, :2, :2, :4]

torch.Size([4, 1024, 12, 64])


tensor([[[[ 0.4362,  0.8201, -2.8764, -0.5829],
          [ 0.1045,  1.4448, -0.6155, -0.8769]],

         [[-0.0139,  1.6894, -2.7798, -0.0927],
          [ 1.6074,  1.9739, -1.5153, -0.3606]]],


        [[[ 0.2431, -0.0582, -1.6742,  0.0144],
          [ 0.8230,  0.9151,  0.1064, -1.5140]],

         [[-0.7463,  0.6405, -1.8554, -0.3869],
          [-0.7816,  0.9120,  0.8533, -0.7739]]]])

In [113]:
# Swap items at x_0, x_1 over last dim (pairwise swap)
k_even = k[:,:,:,0::2]
k_odd = k[:,:,:,1::2]
k_swapped = torch.stack((k_odd, k_even), dim=-1).reshape_as(k)
print(k_swapped.shape)
k_swapped[:2, :2, :2, :4]

torch.Size([4, 1024, 12, 64])


tensor([[[[-0.7154, -1.8227,  0.4927, -1.1770],
          [ 1.3478,  0.0076, -0.6165,  1.7755]],

         [[-1.4143, -1.6687,  1.0009, -1.4753],
          [ 1.0618, -0.2098, -0.1339,  1.1707]]],


        [[[-0.8838, -1.8054, -0.2708, -0.7356],
          [ 0.8396,  0.1831, -0.3262,  1.3368]],

         [[-0.5791, -1.0753, -0.4480, -0.8843],
          [ 0.5428, -1.7920, -0.4998,  1.1403]]]])

In [114]:
q_rot = torch.zeros_like(q)
q_rot[:,:,:,0::2] = cos_trimmed[:,:,:,0::2] * q[:,:,:,0::2] - q_swapped[:,:,:,0::2] * sin_trimmed[:,:,:,0::2]
q_rot[:,:,:,1::2] = cos_trimmed[:,:,:,1::2] * q[:,:,:,1::2] + q_swapped[:,:,:,1::2] * sin_trimmed[:,:,:,1::2]
print(q_rot.shape)
q_rot[:2, :2, :2, :4]

torch.Size([4, 1024, 12, 64])


tensor([[[[ 0.8201,  0.4362, -0.5829, -2.8764],
          [ 1.4448,  0.1045, -0.8769, -0.6155]],

         [[ 0.9244,  1.4141,  1.4036, -2.4011],
          [-0.2861,  2.5295,  0.5028, -1.4742]]],


        [[[-0.0582,  0.2431,  0.0144, -1.6742],
          [ 0.9151,  0.8230, -1.5140,  0.1064]],

         [[ 0.9741,  0.1357,  0.6619, -1.7759],
          [ 1.1504,  0.3451, -1.1097,  0.3093]]]])

In [115]:
k_rot = torch.zeros_like(k)
k_rot[:,:,:,0::2] = cos_trimmed[:,:,:,0::2] * k[:,:,:,0::2] - k_swapped[:,:,:,0::2] * sin_trimmed[:,:,:,0::2]
k_rot[:,:,:,1::2] = cos_trimmed[:,:,:,1::2] * k[:,:,:,1::2] + k_swapped[:,:,:,1::2] * sin_trimmed[:,:,:,1::2]
print(k_rot.shape)
print(k_rot[:2, :2, :2, :4])

torch.Size([4, 1024, 12, 64])
tensor([[[[-1.8227, -0.7154, -1.1770,  0.4927],
          [ 0.0076,  1.3478,  1.7755, -0.6165]],

         [[ 0.2885, -2.1684, -1.7818,  0.0602],
          [-1.0068,  0.3971,  1.0619,  0.5109]]],


        [[[-1.8054, -0.8838, -0.7356, -0.2708],
          [ 0.1831,  0.8396,  1.3368, -0.3262]],

         [[-0.0937, -1.2178, -0.5093, -0.8505],
          [-1.4250, -1.2147,  1.2311,  0.1851]]]])


In [116]:
with torch.no_grad():
    q = q_rot.transpose(1, 2)  # B,nh,T,hs
    k = k_rot.transpose(1, 2)  # B,nh,T,hs
    v = v.transpose(1, 2)  # B,nh,T,hs

    y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

    y = y.transpose(1, 2)  # B,T,nh,hs
    y = y.contiguous()
    y = y.view(B,T,C)

    out = c_proj(y)

    print(out.sum().item())

-20673.5078125


In [ ]:
# Itnore rest of notebook
# --- IGNORE ---

In [ ]:
class CausalSelfAttentionMarcin2(nn.Module):
    """Multiple self-attention heads"""
    def __init__(self, n_head, n_embd):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head

        self.c_q = nn.Linear(n_embd, n_embd)
        self.c_k = nn.Linear(n_embd, n_embd)
        self.c_v = nn.Linear(n_embd, n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)


    def forward(self, x):
        B, T, C = x.size()

        q = self.c_q(x)    # B, T, nh*hs
        k = self.c_k(x)    # B, T, nh*hs
        v = self.c_v(x)    # B, T, nh*hs

        q = q.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        k = k.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        v = v.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        q = q.transpose(1, 2)  # B,nh,T,hs
        k = k.transpose(1, 2)  # B,nh,T,hs
        v = v.transpose(1, 2)  # B,nh,T,hs

        # W_affin = q @ k.mT / k.shape[-1]**0.5  # B,nh,T,hs @ B,nh,hs,T -> B,nh,T,T
        # W_affin = W_affin.masked_fill(self.bias[:,:,:T,:T]==0, float('-inf'))
        # W_affin = torch.softmax(W_affin, dim=-1)  # B,nh,T,T
        # y = W_affin @ v    # B,nh,T,T @ B,nh,T,hs -> B,nh,T,hs
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        y = y.transpose(1, 2)  # B,T,nh,hs
        y = y.contiguous()
        y = y.view(B,T,C)

        out = self.c_proj(y)
        return out

In [ ]:
with torch.no_grad():
    B, T, C = x.size()

    q = c_q(x)    # B, T, nh*hs
    k = c_k(x)    # B, T, nh*hs
    v = c_v(x)    # B, T, nh*hs
    q = q.view(B, T, n_head, C//n_head)  # B,T,nh,hs
    k = k.view(B, T, n_head, C//n_head)  # B,T,nh,hs
    v = v.view(B, T, n_head, C//n_head)  # B,T,nh,hs
    q = q.transpose(1, 2)  # B,nh,T,hs
    k = k.transpose(1, 2)  # B,nh,T,hs
    v = v.transpose(1, 2)  # B,nh,T,hs

    y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

    y = y.transpose(1, 2)  # B,T,nh,hs
    y = y.contiguous()
    y = y.view(B,T,C)

    out = c_proj(y)

    print(out.sum().item())


In [ ]:
csa_m2 = CausalSelfAttentionMarcin2(n_head=n_head, n_embd=n_embd)
csa_m2_state = csa_m2.state_dict()

csa_m2_state['c_q.weight'] = c_attn_W[:n_embd].clone()
csa_m2_state['c_q.bias'] = c_attn_b[:n_embd].clone()
csa_m2_state['c_k.weight'] = c_attn_W[n_embd:2*n_embd].clone()
csa_m2_state['c_k.bias'] = c_attn_b[n_embd:2*n_embd].clone()
csa_m2_state['c_v.weight'] = c_attn_W[2*n_embd:].clone()
csa_m2_state['c_v.bias'] = c_attn_b[2*n_embd:].clone()

csa_m2_state['c_proj.weight'] = c_proj_W.clone()
csa_m2_state['c_proj.bias'] = c_proj_b.clone()
csa_m2.load_state_dict(csa_m2_state)

y_m2 = csa_m2(x)
print(y_m2.shape)
print(y_m2.sum().item())